# Imperative vs declarative panel tutorial 

See: https://panel.holoviz.org/tutorials/intermediate/interactivity.html

## Imperative

In [1]:
import panel as pn
import pandas as pd

pn.extension("tabulator")

data_url = 'https://assets.holoviz.org/panel/tutorials/turbines.csv.gz'

turbines = pn.cache(pd.read_csv)(data_url)

cols = pn.widgets.MultiChoice(
    options=turbines.columns.to_list(), value=['p_name', 't_state', 't_county', 'p_year', 't_manu', 'p_cap'],
    width=500, height=100, name='Columns'
)

So, turbines is a simple dataframe

In [5]:
turbines.head()

,case_id,faa_ors,faa_asn,usgs_pr_id,eia_id,t_state,t_county,t_fips,p_name,p_year,...,retrofit,retrofit_year,t_conf_atr,t_conf_loc,t_img_date,t_img_srce,xlong,ylat,easting,northing
0,3072661,NaN,NaN,5149.0,52161.0,CA,Kern County,6029,251 Wind,1987.0,...,0,NaN,2,3,2018-05-08,Digital Globe,-118.363762,35.077908,-1.317619e+07,4.174474e+06
1,3072695,NaN,NaN,5143.0,52161.0,CA,Kern County,6029,251 Wind,1987.0,...,0,NaN,2,3,2018-05-08,Digital Globe,-118.364410,35.077435,-1.317627e+07,4.174409e+06
2,3072704,NaN,NaN,5146.0,52161.0,CA,Kern County,6029,251 Wind,1987.0,...,0,NaN,2,3,2018-05-08,Digital Globe,-118.364197,35.077644,-1.317624e+07,4.174438e+06
3,3063272,19-028134,2014-WTE-4084-OE,NaN,NaN,IA,Story County,19169,30 MW Iowa DG Portfolio,2017.0,...,0,NaN,3,3,2017-04-24,Digital Globe,-93.430367,42.028233,-1.040062e+07,5.165210e+06
4,3053390,19-028015,2015-WTE-6386-OE,NaN,NaN,IA,Boone County,19015,30 MW Iowa DG Portfolio,2017.0,...,0,NaN,3,3,2017-06-01,Digital Globe,-93.700424,41.977608,-1.043068e+07,5.157626e+06


In [6]:
table = pn.widgets.Tabulator(turbines[cols.value], page_size=5, pagination="remote")

def update_data(event):
    table.value = turbines[event.new]

cols.param.watch(update_data, 'value')

pn.Column(cols, table)

Column
    [0] MultiChoice(height=100, name='Columns', options=['case_id', 'faa_ors', ...], sizing_mode='fixed', value=['p_name', 't_state', ...], width=500)
    [1] Tabulator(page_size=5, pagination='remote', value=              ...)

What is happening here? To my understanding the `.pararm.watch()` method on the `cols` widget binds the `update_data()` callback.   

## Declarative

In [7]:
dfrx = pn.rx(turbines)[cols]

pn.Column(cols, pn.widgets.Tabulator(dfrx, page_size=5, pagination="remote")).servable()

Column
    [0] MultiChoice(height=100, name='Columns', options=['case_id', 'faa_ors', ...], sizing_mode='fixed', value=['t_state', 't_county', ...], width=500)
    [1] Tabulator(page_size=5, pagination='remote', value=      t_state  ...)

## Exercise: Add more Widgets

I think that I need to add a range widget first 

In [53]:
import pandas as pd
import panel as pn

pn.extension("tabulator")

data_url = "https://assets.holoviz.org/panel/tutorials/turbines.csv.gz"

turbines = pn.cache(pd.read_csv)(data_url)

cols = pn.widgets.MultiChoice(
    options=turbines.columns.to_list(),
    value=["p_name", "t_state", "t_county", "p_year", "t_manu", "p_cap"],
    width=500,
    height=100,
    name="Columns",
)
p_year_options = sorted(int(year) for year in turbines.p_year.unique() if not pd.isna(year))
p_year_widget = pn.widgets.Select(value=max(p_year_options), options=p_year_options, name="Year", width=100)

p_cap_bounds = (turbines.p_cap.min(), turbines.p_cap.max())
p_cap_widget = pn.widgets.RangeSlider(value=p_cap_bounds, start=p_cap_bounds[0], end=p_cap_bounds[1], name='Capacity')

dfrx = pn.rx(turbines)
dfrx = dfrx[
    (dfrx['p_year'] == p_year_widget)
    & (dfrx['p_cap'].between(p_cap_widget.param.value_start, p_cap_widget.param.value_end))
][cols]
pn.Column(
    cols, p_year_widget, p_cap_widget, pn.widgets.Tabulator(dfrx, pagination=None, height=500, page_size=5)
)

Column
    [0] MultiChoice(height=100, name='Columns', options=['case_id', 'faa_ors', ...], sizing_mode='fixed', value=['p_name', 't_state', ...], width=500)
    [1] Select(name='Year', options=[1981, 1982, 1983, ...], value=2022, width=100)
    [2] RangeSlider(end=np.float64(1055.6), name='Capacity', start=np.float64(0.05), value=(np.float64(0.05), ..., value_end=np.float64(1055.6), value_start=np.float64(0.05))
    [3] Tabulator(height=500, page_size=5, value=              ...)

I have adapted the code naming a bit to clarify the difference between widgets and column names. Seems quite clear to me.